# Packages & Environments

## REPL and API

Ref: https://pkgdocs.julialang.org/v1/

The above is a very good and easy to read reference. The first chapter pretty much covers all the essentials.

There are two ways to interact with packages and environments. Using the `Pkg` API and using the REPL. In this notebook I'll only talk about the REPL since that seems like a natural way for me to manage packages. But here is a very brief overview of how to use the `Pkg` API should I ever need to dig into it -

```julia
using Pkg
Pkg.add("Example")
```

Will add the `Example` package to the current environment.

In the REPL press the `]` key to enter the package mode and the `backspace` or `delete` key to exit out of it and go back to the REPL. I can type `help` or `?` in pkg mode to get a list of all the commands. To get the details of a particular command I need to do `?<cmd>`.

## Environments

The environment concept is very similar to Python's environment story. There is a default global environment and then I can create local environments that are per project. An environment is just a directory that has two files - 
  * `Project.toml` file that has the dependencies that I explicitly add to my environment.
  * `Manifest.toml` file that has **all** the cascading dependencies based off of the explicit dependencies that I added.

Global environments are called "shared" environments in Julialand.

### Default Shared Environment

To see the global environment, type the `status` command (shortcut is `st`) in the package mode. Here is what my current (Nov 2025) setup outputs -

```
(@v1.12) pkg> st
Status `~/.julia/environments/v1.12/Project.toml`
[6e4b80f9] BenchmarkTools v1.6.3
[7073ff75] IJulia v1.32.1

(@v1.12) pkg>
```

It can be seen that my global environment is `v1.12` in the prompt. I can see the full path is `~/.julia/environments/v1.12/` in the first output line. I can further see that I have two packages installed in this environment - `BenchmarkTools` and `IJulia`.

Here is what the environment directory looks like -

```
╭─avilay ~/.julia/environments/v1.12 via  v1.12.1
🕙 1:20PM
╰─ॐ ls
.rw-r--r-- 8.5k avilay 20 Nov 18:00  Manifest.toml
.rw-r--r--  111 avilay 20 Nov 18:00  Project.toml

╭─avilay ~/.julia/environments/v1.12 via  v1.12.1
🕙 1:20PM
╰─ॐ cat Project.toml
File: Project.toml
[deps]
BenchmarkTools = "6e4b80f9-dd63-53aa-95a3-0cdb28fa8baf"
IJulia = "7073ff75-c697-5162-941a-fcdaad2a7d2a"
```

This file has the hashkey of the pacakge, but not the package version. If I want to know the version I need to peek into the Manifest file.

```
╭─avilay ~/.julia/environments/v1.12 via  v1.12.1
🕙 1:27PM
╰─ॐ cat Manifest.toml | grep -A 4 Benchmark
[[deps.BenchmarkTools]]
deps = ["Compat", "JSON", "Logging", "Printf", "Profile", "Statistics", "UUIDs"]
git-tree-sha1 = "7fecfb1123b8d0232218e2da0c213004ff15358d"
uuid = "6e4b80f9-dd63-53aa-95a3-0cdb28fa8baf"
version = "1.6.3"
```

This tells me that I have Benchmark 1.6.3 installed in this environment.

### Local Environments

To create a local environment I need to run the `activate <dir>` command in pkg mode. `<dir>` is the absolute or relative path of the directory where I want to create the enviornment. Just running this command is not going to actually create the directory and the toml files. Just inside the package mode the prompt will reflect that this is the current environment.

```
(@v1.12) pkg> activate ~/temp/hahaha
Activating new project at `~/temp/hahaha`

(hahaha) pkg> 
```

Once I add a package, it will create the toml files. Unlike Python or Node, the packages themselves are not downloaded on a per environment basis. There is a global cache of packages where all the packages are actually downloaded and compiled. In my current setting it is at `~/.julia/packages` -

```
╭─avilay ~/.julia/packages
🕙 1:51PM
╰─ॐ ls
drwxr-xr-x - avilay 18 Nov 15:46  BenchmarkTools
drwxr-xr-x - avilay 20 Nov 16:24  Example
drwxr-xr-x - avilay 17 Nov 16:05  IJulia
...
```

See [Note](https://pkgdocs.julialang.org/v1/getting-started/#Note-289f56b8414df75).
> If you have the same package (at the same version) installed in multiple environments, the package will only be downloaded and stored on the hard drive once. This makes environments very lightweight and effectively free to create. Using only the default environment with a huge number of packages in it is a common beginners mistake in Julia. Learning how to use environments effectively will improve your experience with Julia packages.

If I have downloaded/cloned a new environment, running `activate` in it is not going to actually download any missing packages into my local package cache. I'll need to `instantiate` the environment for that to happen. I still need to run `activate` before I run `instantiate` though.

If I want to run a command line Julia script without having to activate the environment first I can run -
```
$ julia --project=. myscript.jl
```
This will use the environment in the current directory to run the `myscript.jl` script.

### Custom Shared Environment

I can create other environments under `~/.julia/environments` by using the `--shared` flag with `activate`.
```
(@v1.12) pkg> activate --shared ai
Activating new project at `~/.julia/environments/ai`

(@ai) pkg>
```

Note that shared environments are prefixed with `@` in the prompt. Everything else remains the same as any other environment.

### Temporary aka Throwaway Environments

I can create throwaway environments, called temporary environments to test out some specific thing. Julia will delete the environment once I exit the REPL. However, the packages I install in it will remain in the global cache. See the package management section below on how to clean up unused packages.

```
(@ai) pkg> activate --temp
Activating new project at `/tmp/jl_QNT6Ta`

(jl_QNT6Ta) pkg> add Colors
Resolving package versions...
Installed FixedPointNumbers ─ v0.8.5
Installed ColorTypes ──────── v0.12.1
Installed Colors ──────────── v0.13.1
Installed Reexport ────────── v1.2.2
Updating `/tmp/jl_QNT6Ta/Project.toml`
[5ae59095] + Colors v0.13.1
...
4 dependencies successfully precompiled in 7 seconds. 1 already precompiled.
```

Once I exit out of the REPL, the `/tmp/jl_QNT6Ta` directory will be deleted, but the `Colors` and `ColorTypes` packages will remain.

### Environments and Notebooks

The way to use a specific environment (and its installed packages) is not the same as choosing a Python kernel. In Julia notebooks the kernel is always the same main Julia binary. It is simplest to programattically activate the environment using the Pkg API.

```julia
using Pkg

Pkg.activate("ai"; shared=true)
```

## Packages



A good overview of available packages is in https://julialang.org/packages/. There are multiple ways to discover packages as mentioned in this website. [JuliaHub](https://juliahub.com/ui/Packages) and [Julia Packages](https://juliapackages.com/) seem good. Julia.jl seems defunct. JuliaHub seems to have a bunch of other interesting resources that seem worth checking out. The following commands are to be run in pkg mode in the REPL. Some packages that I can mess around with while learning this stuff are -
  * [Example](https://juliahub.com/ui/Packages/General/Example)
  * [Colors](https://juliahub.com/ui/Packages/General/Colors)

##### Adding
```
pkg> add <pkgone> <pkgtwo> ...
```
Will add the package to the Project.toml file, download the packages in the global cache along with any upstream dependencies, add this and all upstream packages to the Manifest.toml in the currently active environment.

##### Removing
```
pkg> remove <pkgone> <pkgtwo> ...
```
Will remove the specified packages from the currently active environment. This will only work if the specified packages are direct dependencies mentioned in the Projec.toml. To remove a dependent package do -

```
pkg> remove --manifest <pkg>
```

⚠️ This will remove this package from the manifest and all its downstream packages as well!

The `remove` command merely removes the package **reference** from the toml files. The package binary will continue to exist in the package cache. To complete remove the package from the cache as well (and get back some disk space) use the `gc` command. By itself it will delete packages that have been unused for a "significant" amount of time. Running `gc --all` will delete all unferenced packages.

##### Updating
(I have not tried running this yet)

Just running `up` will update all the direct dependencies in the environment. If I want to update a specific package I can say `up <pkgname>`. This will only update the specified package, not its upstream dependencies! If I want to update only the indirect upstream dependencies but not a direct dependency, e.g., if pkgone depends on pkgtwo and pkgthree, but pkgtwo is a direct dependency mentioned in Project.toml, then I want to preserve that as-is, I don't want to update it. I only want to update pkgone and pkgthree, I will do `up --preserve=direct pkgone`. And if I want to update all upstream dependencies, not even preserving the direct ones, I will do `up --preserve=none pkgone`.

##### Pinning
To pin a package to its current version -

```
pkg> pin <pkgname>
```

To "unpin" it -
```
pkg> free <pkgname>
```